In [ ]:
import torch
import torch.nn as nn
import snntorch as snn

device = torch.device("mps")

class SNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(20, 2)
        self.leaky = snn.Leaky(beta=0.9)

    def forward(self, x):
        mem = self.leaky.init_leaky()

        current = self.fc(x)
        spike, mem = self.leaky(current, mem)

        return spike, mem

model = SNN().to(device)

evenX = torch.randint(0, 10, (32, 10)) * 2
oddX = torch.randint(0, 10, (32, 10)) * 2 + 1

evenY = torch.ones(32, dtype=torch.long)
oddY = torch.zeros(32, dtype=torch.long)

x = torch.cat((evenX, oddX), dim=0)
remainderX = x % 2
x = torch.cat((x, remainderX), dim=1).float().to(device)
y = torch.cat ((evenY, oddY), dim=0).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

model.train()

for epoch in range(100):
    optimizer.zero_grad()

    spike, mem = model(x)
    loss = criterion(mem, y)

    loss.backward()
    optimizer.step()

    prediction = mem.argmax(dim=1)
    accuracy = (prediction == y).float().mean()
    print(f"epoch: {epoch + 1}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.2%}")





# @2026-11/17/notebook.ipynb 리뷰
코드를 다시 리뷰해보았습니다.
아무리 봐도 데이터에 정답을 넣어주는 것이 마음에 들지 않습니다. 그래서 한 가지 테스트를 해보려고합니다. 나머지를 넣어주지 않고, 단순히 epoch를 늘려주는 것입니다.

In [5]:
import torch
import torch.nn as nn
import snntorch as snn

device = torch.device("mps")

class SNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2) # 다시 10으로 바뀌어야 함
        self.leaky = snn.Leaky(beta=0.9)

    def forward(self, x):
        mem = self.leaky.init_leaky()

        current = self.fc(x)
        spike, mem = self.leaky(current, mem)

        return spike, mem

model = SNN().to(device)

evenX = torch.randint(0, 10, (32, 10)) * 2
oddX = torch.randint(0, 10, (32, 10)) * 2 + 1

evenY = torch.ones(32, dtype=torch.long)
oddY = torch.zeros(32, dtype=torch.long)

x = torch.cat((evenX, oddX), dim=0).float().to(device)
# remainderX = x % 2
#x = torch.cat((x, remainderX), dim=1).float().to(device)
y = torch.cat ((evenY, oddY), dim=0).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

model.train()

for epoch in range(1001):
    optimizer.zero_grad()

    spike, mem = model(x)
    loss = criterion(mem, y)

    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        prediction = mem.argmax(dim=1)
        accuracy = (prediction == y).float().mean()
        print(f"epoch: {epoch + 1}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.2%}")





epoch: 1, loss: 7.8076, accuracy: 50.00%
epoch: 201, loss: 1.3618, accuracy: 48.44%
epoch: 401, loss: 0.7906, accuracy: 54.69%
epoch: 601, loss: 0.6599, accuracy: 67.19%
epoch: 801, loss: 0.6425, accuracy: 65.62%
epoch: 1001, loss: 0.6387, accuracy: 64.06%


Linear(10, 2)로 수정하고, remainderX를 주석처리하였습니다.
epoch를 1000으로 설정하고, 200번마다 결과를 출력하도록 하였습니다.
결과적으로, 정확도는 여전히 64%입니다.
사실 저는 아직 정확도에 대한 경험이 부족하기에 이 값이 높은지 낮은지 알 수 없습니다.
그래서 GPT에게 물어봤습니다.
- epoch 1000은 큰 값인가? -> epoch보단 성능 향상 여부에 집중하라
- 정확성 65%는 높은가 낮은가 -> baseline과 비교하라. 여기서 baseline은 50%으로 잡을 수 있다. 대충 찍어도 50%으로 수렴.
- 과적합이란? -> 쉽게 비유하자면, 시험의 답을 외움(overfitting) vs 원리를 이해함. 학습 정확도가 높은데 테스트 정확도가 낮다면, 과적합 의심 가능.

현재는 test dataset을 만들지 않았기에 테스트 정확성과 비교 불가능합니다.
그리고 추가적인 지적이, SNN을 충분히 활용하고 있지 않다는 것입니다.
forward에서 반복하지 않고 있기에 step=1이라는 겁니다.

아직 충분한 이해가 부족해서 step이 어떤 영향을 주는지, 정확히 무얼 하는 건지 잘 모릅니다만, 나중에 보면 int v = 1; free(v); 수준의 엉터리 코드일 가능성이 커 보입니다.


In [6]:
import torch
import snntorch as snn

lif = snn.Leaky(beta=0.9)
mem = torch.zeros(1)

current = torch.tensor([0.4])

for t in range(10):
    spike, mem = lif(current, mem)

    print(
        f"t={t+1}, "
        f"mem={mem.item():.4f}, "
        f"spike={spike.item():.0f}"
    )

t=1, mem=0.4000, spike=0
t=2, mem=0.7600, spike=0
t=3, mem=1.0840, spike=1
t=4, mem=0.3756, spike=0
t=5, mem=0.7380, spike=0
t=6, mem=1.0642, spike=1
t=7, mem=0.3578, spike=0
t=8, mem=0.7220, spike=0
t=9, mem=1.0498, spike=1
t=10, mem=0.3448, spike=0


1. 전위막 0 하나
2. 입력 전압 [0.4]
3. step 1 : 0.4 주입
4. step 2 : 0.4 에서 beta 0.9로 인해 0.4 - 0.4*(1-0.9) => 0.36 그리고 새로운 전압 0.4 주입 => 0.36 + 0.4 = 0.76
5. 0.76의 leak, 0.4주입 -> 임계값 초과 -> spike
6. 반복



## Q. 원소가 하나가 아닐 경우 spike 발생시 이것이 다른 원소에 전파되는가? 방향은? 얼마만큼?
## GPT 답변
1. 어디로 전파되는가? → 뉴런 간 연결 구조가 결정해.
2. 얼마나 강하게 전달되는가? → 시냅스 가중치(Weight)가 결정해.
3. 언제 전달되는가? → Time Step과 네트워크의 연산 순서가 결정해.

self.fc = nn.Linear(10, 2)
슬프게도 맥북에 커피를 쏟는 바람에 이 다음에 무얼 하려고 했는지 기억이 안 나네요...